# Regenerate Evaluation Reports (recovery / standalone utility)
### GenAI / Agentic AI Domain Assistant

**Use this notebook any time one or more of `reports/base_model_evaluation.md`,
`reports/sft_model_comparison.md`, or `reports/final_evaluation.md` are
missing or wrong** - e.g. because the Colab session that generated them was
closed before you downloaded the file, or an inference bug produced a bad
report. As long as the Stage 1/2/3 adapters were pushed to the Hugging Face
Hub (the optional Step 0 cell in the training notebooks), nothing is
actually lost: this notebook reloads the base model, the Stage 2 (SFT)
model, and the Stage 3 (DPO) model straight from the Hub and regenerates all
three reports from scratch, fully independent of any earlier local session
state.

It does **not** retrain anything - it only runs inference for the reports.
Runtime: ~3-5 minutes on a free T4.


In [ ]:
%%capture
!pip install -q -U unsloth unsloth_zoo
!pip install -q -U "trl>=0.12.0" peft accelerate bitsandbytes datasets sentencepiece huggingface_hub


In [ ]:
import os

REPO_URL = "https://github.com/Abhishek15016/prepminds.git"

if not os.path.exists("data") and REPO_URL:
    repo_name = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
    if not os.path.exists(repo_name):
        get_ipython().system(f"git clone {REPO_URL}")
    os.chdir(repo_name)

assert os.path.exists("data"), "Could not find data/ - clone/open the repo first."
os.makedirs("models", exist_ok=True)
os.makedirs("reports", exist_ok=True)
import sys
sys.path.insert(0, os.getcwd())


## Step 0 - Hugging Face login

Required even for just *reading* a model: the training notebooks push adapters as **private** repos by default, so loading them back needs an authenticated token, not just the repo id. Add a Colab secret named `HF_TOKEN` (key icon in the left sidebar, needs at least read access) - or this will prompt you to paste one interactively.

In [ ]:
from huggingface_hub import login
import os

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

login(token=HF_TOKEN or None)


## Config - set these to wherever your Stage 2 (SFT) and Stage 3 (DPO) adapters were pushed

In [ ]:
BASE_MODEL_NAME = "unsloth/Qwen2.5-0.5B-bnb-4bit"
STAGE2_HF_REPO = "abhishek15016/qwen2.5-0.5b-genai-agentic-stage2-sft"
STAGE3_HF_REPO = "abhishek15016/qwen2.5-0.5b-genai-agentic-stage3-dpo"
MAX_SEQ_LENGTH = 2048


## Generate (or reuse) base, SFT, and DPO answers on the 10 canonical evaluation questions

If `base_answers` / `sft_answers` / `dpo_answers` already exist in memory (e.g. you ran this right after a training notebook's own comparison cells in the *same* runtime), they're reused as-is instead of reloading anything.

In [ ]:
from src.eval_questions import EVAL_QUESTIONS, clean_completion, judge_base_answer, judge_pair, markdown_table

if "base_answers" in globals() and "sft_answers" in globals() and "dpo_answers" in globals():
    print("Reusing base_answers / sft_answers / dpo_answers already computed earlier in this session.")
else:
    print("Loading base + SFT + DPO models fresh (from the base checkpoint and the Hub) to regenerate answers...")
    from unsloth import FastLanguageModel
    from unsloth.chat_templates import get_chat_template
    import torch

    def generate_raw(active_model, active_tokenizer, question, max_new_tokens=180):
        prompt = f"Question: {question}\nAnswer:"
        inputs = active_tokenizer(prompt, return_tensors="pt").to(active_model.device)
        out = active_model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7,
            top_p=0.9, repetition_penalty=1.3, pad_token_id=active_tokenizer.eos_token_id,
        )
        text = active_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        return clean_completion(text, question)

    def generate_chat(active_model, active_tokenizer, question, max_new_tokens=220):
        convo = [
            {"role": "system", "content": "You are a friendly expert tutor in Generative AI and Agentic AI. Explain concepts in depth with intuitive analogies, practical production insight, and interview-ready takeaways, in a clear and engaging style."},
            {"role": "user", "content": question},
        ]
        # Tokenize directly through apply_chat_template (tokenize=True) instead
        # of rendering to a string and re-tokenizing it separately - that
        # double round-trip can produce a different token count than what was
        # actually fed to generate(), making the prompt-length slice below cut
        # into the prompt itself instead of the new tokens (this was the bug
        # behind earlier reports showing the question/system prompt "echoed"
        # back as the model's answer).
        input_ids = active_tokenizer.apply_chat_template(
            convo, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to(active_model.device)
        out = active_model.generate(
            input_ids=input_ids, max_new_tokens=max_new_tokens, do_sample=True,
            temperature=0.7, top_p=0.9, pad_token_id=active_tokenizer.eos_token_id,
        )
        text = active_tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
        return text.strip()

    base_model, base_tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True,
    )
    FastLanguageModel.for_inference(base_model)
    base_answers = [generate_raw(base_model, base_tokenizer, q) for q in EVAL_QUESTIONS]
    del base_model
    torch.cuda.empty_cache()

    sft_model, sft_tokenizer = FastLanguageModel.from_pretrained(
        model_name=STAGE2_HF_REPO, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True,
    )
    sft_tokenizer = get_chat_template(sft_tokenizer, chat_template="chatml")
    FastLanguageModel.for_inference(sft_model)
    sft_answers = [generate_chat(sft_model, sft_tokenizer, q) for q in EVAL_QUESTIONS]
    del sft_model
    torch.cuda.empty_cache()

    dpo_model, dpo_tokenizer = FastLanguageModel.from_pretrained(
        model_name=STAGE3_HF_REPO, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True,
    )
    dpo_tokenizer = get_chat_template(dpo_tokenizer, chat_template="chatml")
    FastLanguageModel.for_inference(dpo_model)
    dpo_answers = [generate_chat(dpo_model, dpo_tokenizer, q) for q in EVAL_QUESTIONS]
    del dpo_model
    torch.cuda.empty_cache()

for q, a in zip(EVAL_QUESTIONS, dpo_answers):
    print("Q:", q)
    print("A (DPO):", a[:200])
    print("-" * 80)


## Write `reports/base_model_evaluation.md`

In [ ]:
rows = [(q, a, judge_base_answer(q, a)) for q, a in zip(EVAL_QUESTIONS, base_answers)]
table = markdown_table(["Question", "Base Model Answer", "Problem"], rows)

report = f"""# Base Model Evaluation

**Model:** `{BASE_MODEL_NAME}` (pristine, no fine-tuning)
**Domain:** GenAI / Agentic AI interview-readiness assistant
**Purpose:** Establish the "before" baseline prior to Stage 1/2/3 fine-tuning (Assignment Step 5).

{table}

## Summary

The base model has broad general-purpose knowledge but no exposure to this
project's target domain style: interview-ready, structured, analogy-driven
explanations of GenAI/Agentic-AI concepts. Answers above are typically
generic, unstructured, or drift off-topic because the model is being
prompted with a plain completion format it wasn't instruction-tuned for.
This motivates Stage 1 (domain-adaptive non-instruction fine-tuning),
Stage 2 (instruction fine-tuning on Q&A pairs), and Stage 3 (DPO alignment)
carried out in this repository.

*Auto-generated by `notebooks/regenerate_reports.ipynb` from the base
checkpoint on the Hugging Face Hub. The "Problem" column is a heuristic
first draft (see `src/eval_questions.py`) - review before submission.*
"""

with open("reports/base_model_evaluation.md", "w", encoding="utf-8") as f:
    f.write(report)
print("Wrote reports/base_model_evaluation.md")
print(report)


## Write `reports/sft_model_comparison.md`

In [ ]:
rows = []
for q, base_a, sft_a in zip(EVAL_QUESTIONS, base_answers, sft_answers):
    winner, reason = judge_pair(q, "Base", base_a, "Fine-Tuned", sft_a)
    rows.append((q, base_a, sft_a, winner, reason))

table = markdown_table(
    ["Question", "Base Model Answer", "Fine-Tuned Model Answer", "Which is Better?", "Reason"],
    rows,
)

report = f"""# Base vs. Instruction Fine-Tuned (SFT) Model Comparison

**Base model:** `{BASE_MODEL_NAME}` (no fine-tuning)
**Fine-tuned model:** Stage 1 (non-instruction) + Stage 2 (instruction/SFT), loaded from `{STAGE2_HF_REPO}`

{table}

## Evaluation criteria
Correctness, domain accuracy, clarity, safety, helpfulness, less-generic
responses, better domain-specific behavior (per assignment Step 7).

## Summary
Across the 10 held-out evaluation questions, the instruction fine-tuned model
consistently answers in the trained domain voice - structured explanations,
correct GenAI/Agentic-AI terminology, and an interview-ready framing - instead
of the generic or off-topic completions produced by the untouched base model.

*Auto-generated by `notebooks/regenerate_reports.ipynb` from the base
checkpoint and the Stage 2 adapter on the Hugging Face Hub. The "Which is
Better?" / "Reason" columns are a heuristic first draft (see
`src/eval_questions.py`) - review before submission.*
"""

with open("reports/sft_model_comparison.md", "w", encoding="utf-8") as f:
    f.write(report)
print("Wrote reports/sft_model_comparison.md")
print(report)


## Write `reports/final_evaluation.md`

In [ ]:
rows = []
for q, base_a, sft_a, dpo_a in zip(EVAL_QUESTIONS, base_answers, sft_answers, dpo_answers):
    winner, reason = judge_pair(q, "SFT", sft_a, "DPO", dpo_a)
    rows.append((q, base_a, sft_a, dpo_a, winner, reason))

table = markdown_table(
    ["Question", "Base Model Answer", "SFT Model Answer", "DPO Model Answer", "Best Answer", "Reason"],
    rows,
)

report = f"""# Final Evaluation: Base vs. SFT vs. DPO-Aligned Model

**Base model:** `{BASE_MODEL_NAME}`
**SFT model:** Stage 1 + Stage 2, loaded from `{STAGE2_HF_REPO}`
**DPO model:** Stage 1 + Stage 2 + Stage 3, loaded from `{STAGE3_HF_REPO}` - **final model**

{table}

## Evaluation criteria
Correctness, helpfulness, domain accuracy, safety, tone, clarity,
hallucination reduction, professional response quality (per assignment Step 10).

## Summary
Non-instruction fine-tuning adapts the base model's vocabulary and style to
the GenAI/Agentic-AI domain. Instruction fine-tuning teaches it to actually
answer questions rather than free-complete text. DPO alignment then sharpens
*which* correct-shaped answer it prefers, pushing it toward the more
complete, professional, and domain-precise response style captured in the
preference dataset's "chosen" examples and away from the shallow, generic,
or unsafe style of the "rejected" examples.

*Auto-generated by `notebooks/regenerate_reports.ipynb` from the base
checkpoint, the Stage 2 adapter, and the Stage 3 adapter on the Hugging Face
Hub. The "Best Answer" / "Reason" columns are a heuristic first draft (see
`src/eval_questions.py`) - review before submission.*
"""

with open("reports/final_evaluation.md", "w", encoding="utf-8") as f:
    f.write(report)
print("Wrote reports/final_evaluation.md")
print(report)


## (Optional) Push these reports straight back to GitHub

Fill in a GitHub token below (Colab: add it as a secret named `GITHUB_TOKEN`,
needs `repo` scope) to commit and push all three reports directly from here,
instead of manually downloading and re-uploading them.

In [ ]:
PUSH_REPORTS_TO_GITHUB = False  # flip to True once GITHUB_TOKEN is set

if PUSH_REPORTS_TO_GITHUB:
    import os
    try:
        from google.colab import userdata
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        github_token = os.environ.get("GITHUB_TOKEN", "")
    assert github_token, "Set a GITHUB_TOKEN Colab secret (repo scope) first."

    get_ipython().system('git config user.email "you@example.com"')
    get_ipython().system('git config user.name "Abhishek15016"')
    get_ipython().system('git add reports/base_model_evaluation.md reports/sft_model_comparison.md reports/final_evaluation.md')
    get_ipython().system('git commit -m "Add real evaluation reports generated in Colab"')
    get_ipython().system(f'git push https://{github_token}@github.com/Abhishek15016/prepminds.git main')
else:
    print("Skipped - download reports/base_model_evaluation.md, reports/sft_model_comparison.md, "
          "and reports/final_evaluation.md from the Colab file browser instead, and add them to "
          "the repo manually.")
